# Exploring Confluent Flink SQL with Jupyter

This notebook demonstrates how to use `confluent-sql` to interactively explore
Flink tables (Kafka topics) from a Jupyter notebook.

Uses the tables from the Confluent Cloud quickstart (`orders`, `customers`, `products`, `clicks`).

## Setup

```bash
uv add jupyterlab pandas confluent-sql
```

Set your environment variables before starting Jupyter:
- `CONFLUENT_FLINK_API_KEY`
- `CONFLUENT_FLINK_API_SECRET`
- `CONFLUENT_ENV_ID`
- `CONFLUENT_ORG_ID`
- `CONFLUENT_COMPUTE_POOL_ID`
- `CONFLUENT_CLOUD_PROVIDER`
- `CONFLUENT_CLOUD_REGION`
- `CONFLUENT_TEST_DBNAME` (optional)

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from _connection import get_connection

conn = get_connection()
print("Connected to Confluent Flink SQL")

## List available tables

Tables in Flink SQL correspond to Kafka topics in your cluster.

In [ ]:
tables = pd.read_sql("SHOW TABLES", conn)
tables

## Describe a table schema

In [ ]:
schema = pd.read_sql("DESCRIBE `orders`", conn)
schema

## Query data

Snapshot queries return point-in-time results, just like a traditional database.

In [ ]:
orders = pd.read_sql("SELECT * FROM `orders` LIMIT 100", conn)
orders.head(10)

In [ ]:
orders.describe()

## Visualize

In [ ]:
numeric_cols = orders.select_dtypes(include="number").columns.tolist()
if numeric_cols:
    orders[numeric_cols[0]].hist(bins=20)

## Aggregation queries

Flink SQL supports standard SQL aggregations over your Kafka topics.

In [ ]:
revenue = pd.read_sql(
    """
    SELECT
        product_id,
        COUNT(*) AS order_count,
        SUM(price) AS total_revenue
    FROM `orders`
    GROUP BY product_id
    ORDER BY total_revenue DESC
    """,
    conn,
)
revenue.plot.bar(x="product_id", y="total_revenue", title="Revenue by Product")

## Join across tables

Join orders with customers to enrich the data.

In [ ]:
enriched = pd.read_sql(
    """
    SELECT
        o.order_id,
        c.name AS customer_name,
        o.product_id,
        o.price
    FROM `orders` o
    JOIN `customers` c ON o.customer_id = c.customer_id
    LIMIT 20
    """,
    conn,
)
enriched

## Cleanup

In [ ]:
conn.close()
print("Connection closed.")